# Learning a solution map

This notebook accompanies Exercise 7.4 and prepares for the chapter on solving equations. For

\[
u_t=-\mu u,\qquad u(0;\mu)=1,
\]

the exact solution is $u(t;\mu)=e^{-\mu t}$. We learn the machine $(t,\mu)\mapsto u(t;\mu)$ from solution values.

## Outline

1. [Data and domains](#donnees-solution)
2. [Polynomial, SVR and MLP](#machines-solution)
3. [Interpolation and extrapolation](#erreurs-solution)
4. [Parameter sensitivity](#sensibilite-solution)
5. [From data to the equation](#equation-solution)

In [ ]:
import jax
import jax.numpy as jnp
import flax
from flax import nnx
import optax
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR

jax.config.update("jax_enable_x64", True)
print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, scikit-learn {sklearn.__version__}"
)

<a id="donnees-solution"></a>
## 1. Data and domains

Construct training and validation data for $t\in[0,1]$ and $\mu\in[0,2.4]$. Reserve $\mu\in(2.4,3]$ for extrapolation. The exact function then allows evaluation on grids much finer than the data.

In [ ]:
# Complete this section.

<a id="machines-solution"></a>
## 2. Polynomial, SVR and MLP

Fit a polynomial machine, Gaussian-kernel SVR and an MLP in turn. All three machines use the same solution values but different function classes and learning procedures.

In [ ]:
class MLP(nnx.Module):
    def __init__(self, largeur, *, rngs):
        self.W1 = nnx.Linear(2, largeur, rngs=rngs)
        self.W2 = nnx.Linear(largeur, largeur, rngs=rngs)
        self.W3 = nnx.Linear(largeur, 1, rngs=rngs)

    def __call__(self, x):
        x = nnx.tanh(self.W1(x))
        x = nnx.tanh(self.W2(x))
        return self.W3(x).reshape(-1)


def perte_mlp(machine, x, z):
    return jnp.mean((machine(x) - z) ** 2)


@nnx.jit
def pas_mlp(machine, optimizer, x, z):
    valeur, gradient = nnx.value_and_grad(perte_mlp)(machine, x, z)
    optimizer.update(machine, gradient)
    return valeur


def entrainer_mlp(machine, x, z, *, alpha=2e-3, iterations=3000):
    optimizer = nnx.Optimizer(machine, optax.adam(alpha), wrt=nnx.Param)
    x = jnp.asarray(x)
    z = jnp.asarray(z)
    historique = []
    for k in range(iterations):
        valeur = pas_mlp(machine, optimizer, x, z)
        if k % 100 == 0:
            historique.append(float(valeur))
    return historique

In [ ]:
# Complete this section.

<a id="erreurs-solution"></a>
## 3. Interpolation and extrapolation

Compute approximate $L^2$ and $L^\infty$ errors on both grids. Plot exact and learned solutions in the interpolation domain, then examine $\mu>2.4$ separately.

In [ ]:
# Complete this section.

<a id="sensibilite-solution"></a>
## 4. Parameter sensitivity

Use automatic differentiation to compute

\[
\frac{\partial\Phi}{\partial\mu}(t,\mu)
\]

for the MLP. Compare it with the exact sensitivity $-t e^{-\mu t}$.

In [ ]:
# Complete this section.

<a id="equation-solution"></a>
## 5. From data to the equation

The preceding machine sees only pairs $((t,\mu),u)$. It uses neither the equation $u_t+\mu u=0$ nor the initial condition. A PINN method instead adds equation and boundary-condition residuals to the loss function. The next chapter examines this change of framework and its difficulties, without assuming that an MLP automatically solves a differential equation.

## Summary

Learning a solution map already reduces evaluation cost: once trained, the machine rapidly supplies $u(t;\mu)$ and its derivatives. However, interpolation, extrapolation and sensitivity are distinct criteria. Loss on the data does not automatically control them.